In [1]:
import os
import sys
from pathlib import Path

# Ajusta al root del repo Bluegrey si hace falta
ROOT = Path.cwd()
if (ROOT / "tools").exists():
    repo_root = ROOT
else:
    # si estás dentro de research/, sube un nivel
    repo_root = ROOT.parent

sys.path.append(str(repo_root))
print("Repo root:", repo_root)

Repo root: /Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey


#### Crea el ingestor y descarga
- El script ya trae una clase lista: PolygonIngestor.
- Descubre pares FX líquidos (fetch_liquid_fx_tickers)
- Descarga velas de 1 minuto (list_aggs)
- Guarda en ArcticDB (write/update)

In [2]:
from tools.download_history_polygon import PolygonIngestor

ing = PolygonIngestor()

# Ver algunos pares que detecta
pairs = ing.fetch_liquid_fx_tickers()
print("Total pares:", len(pairs))
print("Ejemplo:", pairs[:10])

# Descargar solo unos pocos para prueba rápida
for t in pairs[:3]:
    ing.download_ticker(t, start_year=2024)

2026-04-06 19:40:32,176 [INFO] Querying Polygon for FX pairs (Liquid Filter Active)...
2026-04-06 19:40:33,409 [INFO] Filter Complete: Reduced universe from 1000+ to 167 high-quality pairs.
Total pares: 167
Ejemplo: ['C:AUDCAD', 'C:AUDCHF', 'C:AUDEUR', 'C:AUDGBP', 'C:AUDHKD', 'C:AUDJPY', 'C:AUDMXN', 'C:AUDNOK', 'C:AUDNZD', 'C:AUDSEK']
✅ Created C:AUDCAD: 835779 bars.
✅ Created C:AUDCHF: 838354 bars.
✅ Created C:AUDEUR: 839265 bars.


In [3]:
from src.store import DataStore
import src.config as config

store = DataStore(config.LIBS["fx_min"])  # normalmente "fx.min"
print("OK store:", config.LIBS["fx_min"])

2026-04-06 19:42:36,250 [INFO] 🗄️ Connected to ArcticDB at: lmdb:///Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey/data/arctic_db?map_size=10GB
OK store: fx.min


20260406 19:42:36.250237 6913109 W arcticdb | LMDB path at /Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey/data/arctic_db/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[15478]


In [4]:
# Verifica que se guardó en ArcticDB

from src.store import DataStore
import src.config as config

store = DataStore(config.LIBS["fx_min"])  # normalmente "fx.min"
print("OK store:", config.LIBS["fx_min"])

2026-04-06 19:43:14,437 [INFO] 🗄️ Connected to ArcticDB at: lmdb:///Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey/data/arctic_db?map_size=10GB
OK store: fx.min


20260406 19:43:14.436628 6913109 W arcticdb | LMDB path at /Users/marina/Documents/jordi/projects/Trading/bluegrey/0. Discover/Bluegrey/data/arctic_db/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[15478]


In [6]:
df = store.load("C:AUDCAD", start_date="2024-01-01")
print(df.head())
print(df.tail())
print("Rows:", len(df))

                         open      high       low     close  volume    vwap
timestamp                                                                  
2024-01-01 00:32:00  0.902200  0.902339  0.902200  0.902339       2  0.9023
2024-01-01 00:33:00  0.902100  0.902100  0.901461  0.901461       2  0.9018
2024-01-01 00:34:00  0.902000  0.902100  0.901961  0.901961       3  0.9020
2024-01-01 08:31:00  0.902100  0.902100  0.902100  0.902100       1  0.9021
2024-01-01 09:59:00  0.902445  0.902445  0.902000  0.902000       3  0.9022
                         open      high     low    close  volume    vwap
timestamp                                                               
2026-04-06 17:35:00  0.962310  0.962501  0.9619  0.96227     226  0.9623
2026-04-06 17:36:00  0.962290  0.962457  0.9618  0.96236     181  0.9623
2026-04-06 17:37:00  0.962350  0.962528  0.9620  0.96235     177  0.9624
2026-04-06 17:38:00  0.962360  0.962501  0.9619  0.96231     197  0.9623
2026-04-06 17:39:00  0.962409 